In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from difflib import get_close_matches

In [ ]:
DATA_PATH = Path("data/netflix_titles.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

In [ ]:
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
print("Dataset shape:", df.shape)

print("\nDataset columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.fillna("")

df["title"] = df["title"].astype(str)

df["release_year"] = pd.to_numeric(
    df["release_year"],
    errors="coerce"
)

print("Missing values handled.")

In [ ]:
TEXT_COLUMNS = [
    "director",
    "cast",
    "listed_in",
    "description",
    "country"
]

In [ ]:
def clean_text(value):

    if isinstance(value, list):
        value = " ".join(value)

    value = re.sub(
        r"[^a-zA-Z0-9 ]",
        " ",
        str(value)
    )

    return value.lower()

In [ ]:
def build_content(row):

    parts = [
        clean_text(row[col])
        for col in TEXT_COLUMNS
    ]

    return " ".join(parts)

In [ ]:
df["content"] = df.apply(
    build_content,
    axis=1
)

df[["title", "content"]].head()

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_features=45000,
    sublinear_tf=True
)

In [ ]:
tfidf_matrix = vectorizer.fit_transform(
    df["content"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
def calculate_similarity(movie_index):

    similarities = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).ravel()

    return similarities

In [ ]:
def normalize_title(title):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(title).lower()
    )

In [ ]:
df["search_title"] = df["title"].map(
    normalize_title
)

title_to_indices = df.groupby(
    "search_title"
).indices

In [ ]:
def find_title(title):

    key = normalize_title(title)

    if not key:
        return None

    indices = title_to_indices.get(key)

    if indices is None:
        return None

    return df.iloc[int(indices[0])]

In [ ]:
movie = find_title("The Dark Knight")

if movie is not None:
    display(movie)
else:
    print("Movie not found.")

In [ ]:
def search_titles(query, limit=8):

    q = normalize_title(query)

    if not q:
        return []

    titles = df[
        ["title", "type", "release_year"]
    ].drop_duplicates("title")

    exact = titles[
        titles["title"]
        .map(normalize_title)
        .str.contains(q, regex=False)
    ]

    return exact.head(limit)

In [ ]:
search_titles("dark")

In [ ]:
def recommend(
    title,
    limit=12,
    kind="All",
    genre="All"
):

    key = normalize_title(title)

    indices = title_to_indices.get(key)

    if indices is None:
        return pd.DataFrame()

    source_idx = int(indices[0])

    similarities = calculate_similarity(source_idx)

    ranked = similarities.argsort()[::-1]

    recommendations = []

    for idx in ranked:

        # Exclude the selected movie
        if idx == source_idx:
            continue

        # Ignore zero-similarity results
        if similarities[idx] <= 0:
            continue

        row = df.iloc[int(idx)]

        # Filter by movie or TV show
        if kind != "All" and row["type"] != kind:
            continue

        # Filter by genre
        row_genres = str(row["listed_in"])

        if genre != "All" and genre.lower() not in row_genres.lower():
            continue

        recommendations.append({
            "title": row["title"],
            "type": row["type"],
            "year": row["release_year"],
            "genres": row["listed_in"],
            "description": row["description"],
            "similarity_score": round(
                float(similarities[idx]) * 100,
                1
            )
        })

        if len(recommendations) >= max(
            1,
            min(int(limit), 30)
        ):
            break

    return pd.DataFrame(recommendations)

In [ ]:
recommendations = recommend(
    title="The Dark Knight",
    limit=10
)

display(recommendations)

In [ ]:
recommendations = recommend(
    title="The Dark Knight",
    limit=10,
    kind="Movie",
    genre="Action"
)

display(recommendations)

In [ ]:
def available_genres():

    genres = set()

    for value in df["listed_in"].astype(str):

        genres.update(
            x.strip()
            for x in value.split(",")
            if x.strip()
        )

    return sorted(genres)

In [ ]:
genres = available_genres()

print("Total genres:", len(genres))

print(genres)

In [ ]:
stats = {
    "total_titles": len(df),

    "movies": (
        df["type"] == "Movie"
    ).sum(),

    "tv_shows": (
        df["type"] == "TV Show"
    ).sum(),

    "total_genres": len(available_genres())
}

stats

In [ ]:
def featured_titles(limit=8):

    sample = (
        df[df["description"].astype(str).str.len() > 40]
        .sort_values(
            ["release_year", "title"],
            ascending=[False, True]
        )
        .head(limit)
    )

    return sample[
        ["title", "type", "release_year", "listed_in"]
    ]

In [ ]:
display(featured_titles())